# Build a Model Cascade with Pareto Analysis

Companion notebook for the [Cost Modeling and Optimization](https://codemia.ai/courses/applied_ai_systems/cost_modeling_and_optimization) lesson on Codemia.

**What you will build**: A two-tier cascade with a confidence-threshold router, the Pareto frontier its thresholds trace through cost-accuracy space, and the one number that justifies shipping it: the spend you save at equal quality.

**Why from scratch**: The cascade formula is three lines and every write-up gets the same three lines wrong in the same two ways. Building the accounting yourself is what surfaces the cheap call you paid for before escalating, and the price of the confidence signal you routed on.

**References**:
- Chen, Zaharia and Zou, FrugalGPT (2023), https://arxiv.org/abs/2305.05176
- Wang et al., Self-Consistency Improves Chain of Thought Reasoning (2022), https://arxiv.org/abs/2203.11171
- Guo et al., On Calibration of Modern Neural Networks (2017), https://arxiv.org/abs/1706.04599

**Time**: ~45 minutes  **GPU required**: no

## Setup

numpy and matplotlib. The notebook runs end to end with a simulated pair of models,
which is what makes every cell reproducible and free; the final section swaps in real
gpt-4o-mini and gpt-4o calls behind an explicit flag.

Be clear-eyed about what the simulation can and cannot tell you. It cannot tell you
how accurate gpt-4o-mini is on your task. It can tell you how a cascade's economics
respond to accuracy gaps, price ratios and confidence quality, because those are
arithmetic, and arithmetic does not care where the accuracy numbers came from.

In [ ]:
%pip install -q numpy matplotlib openai

In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt

# Opt in deliberately. The live sweep calls gpt-4o once per escalated item, so a
# stray True here is a real invoice. The presence of a key is not consent.
RUN_LIVE = False

rng = np.random.default_rng(0)
print('live sweep enabled:', RUN_LIVE)
print('API key present   :', bool(os.environ.get('OPENAI_API_KEY')))

## The economics, written out once

A cascade sends every request to the cheap model, keeps the answer when the router is
confident, and escalates otherwise. Let `e` be the escalation rate, `c_w` the cost of
one weak call and `c_s` the cost of one strong call. Then

```
cost_per_request = k * c_w + e * c_s
```

Two terms in that line are where the mistakes live.

**The weak call is never refunded.** You pay `c_w` on every request, including the
ones you escalate. The naive version, `(1 - e) * c_w + e * c_s`, quietly assumes
escalation gets your money back. At `e = 1` the real cascade costs `c_w + c_s`, which
is *more* than always calling the strong model, so a cascade tuned on the naive
formula will route itself into losing money and report a saving while doing it.

**The confidence signal has a price, and it is the `k`.** If you get confidence by
sampling the weak model `k` times, the cheap tier costs `k * c_w`, and a 5-sample
self-consistency signal makes it five times less cheap. That is an attack on the only
property that made the cheap tier worth having, which is why `k` is an argument to
the cost function below and a swept variable everywhere after it.

In [ ]:
# Published per-million-token prices, checked 2026-10-05. Verify before quoting
# these: vendor pricing moves, and the savings number below moves with it. The
# conclusions here depend on the RATIO c_s / c_w far more than on either absolute
# price, which is why there is a ratio sweep near the end that finds its own
# breakeven instead of trusting today's figures.
PRICES = {
    'gpt-4o-mini': {'in': 0.15, 'out': 0.60},
    'gpt-4o': {'in': 2.50, 'out': 10.00},
}
TOKENS = {'in': 220, 'out': 90}   # one benchmark item, prompt and answer


def call_cost(model):
    p = PRICES[model]
    return (TOKENS['in'] * p['in'] + TOKENS['out'] * p['out']) / 1e6


C_WEAK, C_STRONG = call_cost('gpt-4o-mini'), call_cost('gpt-4o')
print(f'one weak call   ${C_WEAK:.6f}')
print(f'one strong call ${C_STRONG:.6f}')
print(f'price ratio     {C_STRONG / C_WEAK:.1f}x')

In [ ]:
def cascade_cost(escalation_rate, k=1, c_w=C_WEAK, c_s=C_STRONG):
    """Cost per request. The k * c_w term is the confidence signal you bought."""
    return k * c_w + escalation_rate * c_s


def verdict(cost, baseline=None):
    """Say cheaper or more expensive in words. A signed percentage sitting next to
    the word 'saving' gets read backwards about half the time, author included."""
    baseline = C_STRONG if baseline is None else baseline
    delta = 1.0 - cost / baseline
    return f'{abs(delta):5.1%} {"cheaper" if delta > 0 else "more expensive"}'


print(f'{"configuration":34s} {"cost/request":>13s}  vs always-strong')
for label, e, k in [('always weak, no confidence', 0.0, 1),
                    ('always weak, 5-sample confidence', 0.0, 5),
                    ('cascade, 30% escalation, k=1', 0.30, 1),
                    ('cascade, 30% escalation, k=5', 0.30, 5),
                    ('cascade, 100% escalation, k=1', 1.00, 1),
                    ('always strong (the baseline)', None, None)]:
    cost = C_STRONG if e is None else cascade_cost(e, k)
    print(f'{label:34s} {"$" + format(cost, ".6f"):>13s}  {verdict(cost)}')
print()
print('The 100% row is the warning: a cascade that escalates everything costs more')
print('than the strong model alone, because the weak call is not refunded.')

## The benchmark

Arithmetic word problems with operator precedence, generated with known answers. Two
properties make them useful here: correctness is checkable without a judge model, so
accuracy is a measurement rather than an opinion, and difficulty is controllable, so
the benchmark spans the range where a cascade's routing decisions actually differ.

This benchmark is also deliberately not saturated. A benchmark both tiers ace has
nothing to teach a router, and that is the most common reason a cascade evaluation
comes back reporting that the threshold does not matter.

In [ ]:
def make_item(rng):
    """An expression whose value depends on precedence, plus its true answer."""
    a, b, c = rng.integers(2, 20, 3)
    d = rng.integers(2, 10)
    form = rng.integers(0, 3)
    if form == 0:
        text, answer = f'{a} + {b} * {c}', a + b * c
    elif form == 1:
        text, answer = f'({a} + {b}) * {c} - {d}', (a + b) * c - d
    else:
        text, answer = f'{a} * {b} - {c} * {d}', a * b - c * d
    # Nesting depth and magnitude both push difficulty up, which is the handle the
    # simulator uses below and also roughly what makes these harder in practice.
    difficulty = min(1.0, (0.25 * form + abs(answer) / 400 + rng.uniform(0, 0.35)))
    return {'text': text, 'answer': int(answer), 'difficulty': float(difficulty)}


ITEMS = [make_item(rng) for _ in range(600)]
print(f'{len(ITEMS)} items, difficulty mean '
      f'{np.mean([i["difficulty"] for i in ITEMS]):.2f}')
for item in ITEMS[:4]:
    print(f'  {item["text"]:24s} = {item["answer"]:6d}   '
          f'difficulty {item["difficulty"]:.2f}')

## The simulated model pair

Each tier is a probability of being correct that falls off with item difficulty, with
the strong tier's curve shifted up. When a tier is wrong it picks a distractor, and
the distractor distribution is skewed on purpose: real models make the *same* mistake
repeatedly, which is what stops self-consistency from being a free oracle.

The parameters below are a stated assumption, not a measurement. Nothing downstream
asserts that gpt-4o-mini is accurate to any particular degree on anything; what the
notebook measures is how a cascade's economics behave given a gap of this shape.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


A_WEAK, A_STRONG, SLOPE = 3.2, 4.8, 4.0

# Wrong answers come from a short, skewed list of plausible slips: off-by-one,
# precedence applied left to right, a sign error. The skew is the point. If wrong
# answers were uniform noise, k samples would almost never agree on a wrong answer
# and self-consistency would detect every error for free.
DISTRACTOR_WEIGHTS = np.array([0.45, 0.25, 0.15, 0.10, 0.05])
DISTRACTOR_OFFSETS = [1, -1, 7, -7, 13]


def answer_once(item, tier, rng):
    """Return one sampled answer from the given tier."""
    a = A_WEAK if tier == 'weak' else A_STRONG
    if rng.random() < sigmoid(a - SLOPE * item['difficulty']):
        return item['answer']
    which = rng.choice(len(DISTRACTOR_WEIGHTS), p=DISTRACTOR_WEIGHTS)
    return item['answer'] + int(DISTRACTOR_OFFSETS[which])


probe = ITEMS[0]
print('true answer:', probe['answer'])
print('10 weak samples  :', [answer_once(probe, 'weak', rng) for _ in range(10)])
print('10 strong samples:', [answer_once(probe, 'strong', rng) for _ in range(10)])

## Two confidence signals, one free and one expensive

**Self-consistency.** Sample the weak tier `k` times, take the majority answer, and
let confidence be the share of samples that agreed. It costs `k` calls instead of
one, and it raises accuracy as a side effect, because a majority vote over `k`
samples is a better estimator than a single sample.

**A token-probability proxy.** One call, with confidence from the model's own
reported likelihood. Free, but noisier, and biased in a direction the calibration
section will measure rather than assume.

Which belongs in a cascade is an empirical question with three candidates, so `k=3`
and `k=5` are carried alongside the free signal all the way through. The usual
assumption is that the better signal wins; hold that thought until the frontier,
because the better signal is also the one attacking its own tier's cheapness.

In [ ]:
def weak_self_consistency(item, k, rng):
    """Majority answer over k samples, plus the agreement share as confidence."""
    samples = [answer_once(item, 'weak', rng) for _ in range(k)]
    values, counts = np.unique(samples, return_counts=True)
    best = int(np.argmax(counts))
    return int(values[best]), counts[best] / k


def weak_logprob(item, rng):
    """One sample, and a noisy read on how likely it is to be right."""
    answer = answer_once(item, 'weak', rng)
    true_p = sigmoid(A_WEAK - SLOPE * item['difficulty'])
    # Deliberately biased upward: a model's own confidence is informative and
    # usually optimistic. The calibration section measures that bias rather than
    # taking this comment's word for it.
    reported = np.clip(true_p + rng.normal(0.08, 0.12), 0.02, 0.99)
    return answer, float(reported)


for k in (3, 5):
    ans, conf = weak_self_consistency(ITEMS[0], k, rng)
    print(f'self-consistency k={k}: answer {ans}, confidence {conf:.2f}')
ans, conf = weak_logprob(ITEMS[0], rng)
print(f'logprob proxy      : answer {ans}, confidence {conf:.2f}')

## Run every tier over the benchmark once

One pass, recording for every item what each configuration answered, what confidence
the router would have seen, and whether the strong tier got it right. Everything
after this is analysis of this one table.

That ordering is the point, not an implementation detail. Thresholds are swept over a
recorded table, so forty-one operating points cost nothing extra; sweeping by
re-running the models at every threshold is how a cost study ends up costing more
than the savings it finds.

In [ ]:
K_LIST = (3, 5)
CONFIGS = [('sc3', 'self-consistency k=3', 3),
           ('sc5', 'self-consistency k=5', 5),
           ('lp', 'logprob proxy k=1', 1)]

run = np.random.default_rng(42)
table = []
for item in ITEMS:
    row = {'truth': item['answer'], 'difficulty': item['difficulty']}
    for k in K_LIST:
        a, c = weak_self_consistency(item, k, run)
        row[f'sc{k}_answer'] = a
        row[f'sc{k}_conf'] = c
        row[f'sc{k}_ok'] = a == item['answer']
    a, c = weak_logprob(item, run)
    row['lp_answer'], row['lp_conf'], row['lp_ok'] = a, c, a == item['answer']
    row['strong_ok'] = answer_once(item, 'strong', run) == item['answer']
    table.append(row)

n = len(table)
acc = {key: sum(r[f'{key}_ok'] for r in table) / n for key, _, _ in CONFIGS}
acc_strong = sum(r['strong_ok'] for r in table) / n

print(f'{"configuration":22s} {"accuracy":>9s} {"cost/req":>11s}   (no routing yet)')
print(f'{"weak, 1 sample":22s} {acc["lp"]:9.3f} {"$" + format(C_WEAK, ".6f"):>11s}')
for k in K_LIST:
    print(f'{f"weak, {k} samples":22s} {acc[f"sc{k}"]:9.3f} '
          f'{"$" + format(k * C_WEAK, ".6f"):>11s}')
print(f'{"strong, 1 sample":22s} {acc_strong:9.3f} {"$" + format(C_STRONG, ".6f"):>11s}')
print()
print(f'Majority voting alone bought {acc["sc5"] - acc["lp"]:+.3f} accuracy at k=5 and')
print(f'{acc["sc3"] - acc["lp"]:+.3f} at k=3, before a single routing decision was made.')

## Calibration, before any threshold

A confidence threshold is only meaningful if confidence means something. The test is
blunt: bucket items by reported confidence and ask what share of each bucket was
actually right. On a calibrated signal those two columns match, and the sign of the
gap tells you which way the signal lies.

Run this before the frontier, not after. A miscalibrated signal still produces a
smooth, entirely plausible cost-accuracy curve; the curve simply is not about what
the axis label claims. One cell distinguishes those two cases.

In [ ]:
def calibration(rows, conf_key, ok_key, bins=8):
    edges = np.linspace(0, 1, bins + 1)
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        group = [r for r in rows
                 if lo <= r[conf_key] < hi or (hi == 1.0 and r[conf_key] == 1.0)]
        if not group:
            continue
        out.append({'lo': lo, 'hi': hi, 'n': len(group),
                    'mean_conf': float(np.mean([r[conf_key] for r in group])),
                    'accuracy': float(np.mean([r[ok_key] for r in group]))})
    return out


def ece(buckets, total):
    """Expected calibration error: mean gap, weighted by bucket population."""
    return sum(b['n'] / total * abs(b['mean_conf'] - b['accuracy']) for b in buckets)


ECE = {}
for key, label, _ in CONFIGS:
    buckets = calibration(table, f'{key}_conf', f'{key}_ok')
    ECE[key] = ece(buckets, n)
    print(f'\n{label}:  ECE = {ECE[key]:.3f}, {len(buckets)} populated buckets')
    print(f'  {"confidence":>12s} {"n":>5s} {"mean conf":>10s} '
          f'{"accuracy":>9s} {"gap":>7s}')
    for b in buckets:
        # Under 30 items a bucket rate is too noisy to read; flagged, not hidden.
        flag = '  (thin)' if b['n'] < 30 else ''
        print(f'  {b["lo"]:.2f}-{b["hi"]:.2f}   {b["n"]:5d} {b["mean_conf"]:10.3f} '
              f'{b["accuracy"]:9.3f} {b["accuracy"] - b["mean_conf"]:+7.3f}{flag}')

In [ ]:
# The sign of the gap is the finding, so print it as a direction rather than
# leaving the reader to infer it from a column of signed floats.
print('bias direction, weighted across populated buckets:')
for key, label, _ in CONFIGS:
    buckets = calibration(table, f'{key}_conf', f'{key}_ok')
    signed = sum(b['n'] / n * (b['accuracy'] - b['mean_conf']) for b in buckets)
    word = 'under-confident' if signed > 0 else 'over-confident'
    print(f'  {label:22s} ECE {ECE[key]:.3f}   mean signed gap {signed:+.3f}  {word}')

print()
print('Granularity matters as much as the gap. An agreement share over k samples can')
print('only take k + 1 distinct values, so a coarse signal offers a short menu of')
print('reachable operating points no matter how finely the threshold is swept.')
for k in K_LIST:
    vals = sorted(set(r[f'sc{k}_conf'] for r in table))
    print(f'  k={k}: {len(vals)} distinct confidence values  '
          f'{[round(v, 2) for v in vals]}')
print(f'  logprob: {len(set(round(r["lp_conf"], 4) for r in table))} '
      f'distinct values (effectively continuous)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (key, label, _) in zip(axes, CONFIGS):
    buckets = calibration(table, f'{key}_conf', f'{key}_ok')
    sizes = [max(14.0, 340.0 * b['n'] / n) for b in buckets]
    ax.plot([0, 1], [0, 1], '--', color='grey', lw=1, label='perfect calibration')
    ax.scatter([b['mean_conf'] for b in buckets], [b['accuracy'] for b in buckets],
               s=sizes, zorder=3, label='bucket (area = share of items)')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel('reported confidence')
    ax.set_ylabel('measured accuracy')
    ax.set_title(f'{label}\nECE {ECE[key]:.3f}', fontsize=10)
    ax.legend(loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

## The router, swept

Sweep the threshold from 0 to 1. At each setting, measure the share of items that
escalated, the cascade's accuracy, and its cost per request using the formula from
the top, `k` included.

Escalated items take the strong tier's answer. Kept items take the weak tier's
answer, *including the ones it got wrong*, which is the whole content of the accuracy
column and the reason a cascade is not simply a cheaper strong model.

In [ ]:
def sweep(rows, key, k, label):
    out = []
    for threshold in np.linspace(0.0, 1.0, 41):
        escalate = [r[f'{key}_conf'] < threshold for r in rows]
        correct = [r['strong_ok'] if e else r[f'{key}_ok']
                   for r, e in zip(rows, escalate)]
        rate = float(np.mean(escalate))
        out.append({'threshold': float(threshold), 'escalation': rate,
                    'accuracy': float(np.mean(correct)),
                    'cost': cascade_cost(rate, k=k), 'k': k, 'label': label})
    return out


SWEEPS = {key: sweep(table, key, k, label) for key, label, k in CONFIGS}

for key, label, _ in CONFIGS:
    print(f'\n{label}')
    print(f'{"thresh":>7s} {"escalated":>10s} {"accuracy":>9s} {"cost/req":>10s}')
    for row in SWEEPS[key][::5]:
        print(f'{row["threshold"]:7.2f} {row["escalation"]:10.1%} '
              f'{row["accuracy"]:9.3f} ${row["cost"]:9.6f}')

## The Pareto analysis

A threshold sweep is a list of candidate systems, not a frontier. A point belongs on
the frontier only if nothing else is both at least as accurate and at least as cheap;
everything else is **dominated**, meaning there is a strictly better system sitting in
the same list and no reason to ever ship it.

Worth computing rather than eyeballing off the chart, because a sweep usually contains
dominated points and they cluster at the aggressive end that looks most impressive.
Escalating an item the weak tier got right and the strong tier gets wrong costs money
*and* accuracy, so past some threshold the curve bends backwards, and reading it as if
it rises monotonically recommends a system that is strictly worse than a cheaper one.

In [ ]:
def pareto(points):
    """Points that nothing else beats on both axes at once.

    Exact ties are collapsed first. Without that, every threshold below a signal's
    lowest observed confidence escalates nothing, so they are all the same system at
    the same cost and accuracy, and none of them dominates any other. They would all
    survive the test and pad the frontier with copies of one point.
    """
    seen, unique = set(), []
    for p in points:
        fingerprint = (round(p['cost'], 9), round(p['accuracy'], 9))
        if fingerprint not in seen:
            seen.add(fingerprint)
            unique.append(p)
    keep = [p for p in unique
            if not any(q['cost'] <= p['cost'] and q['accuracy'] >= p['accuracy']
                       and (q['cost'] < p['cost'] or q['accuracy'] > p['accuracy'])
                       for q in unique)]
    return sorted(keep, key=lambda d: d['cost'])

In [ ]:
BASELINES = ([{'label': 'always weak, 1 sample', 'cost': C_WEAK,
               'accuracy': acc['lp'], 'escalation': 0.0, 'k': 1}]
             + [{'label': f'always weak, {k} samples', 'cost': k * C_WEAK,
                 'accuracy': acc[f'sc{k}'], 'escalation': 0.0, 'k': k}
                for k in K_LIST]
             + [{'label': 'always strong', 'cost': C_STRONG,
                 'accuracy': acc_strong, 'escalation': 0.0, 'k': 0}])

ALL_POINTS = [p for key, _, _ in CONFIGS for p in SWEEPS[key]] + BASELINES
FRONTIER = pareto(ALL_POINTS)
distinct = len({(round(p['cost'], 9), round(p['accuracy'], 9)) for p in ALL_POINTS})
print(f'{len(ALL_POINTS)} candidate systems, {distinct} of them distinct, '
      f'{len(FRONTIER)} on the Pareto frontier, {distinct - len(FRONTIER)} dominated')
print()
step = max(1, len(FRONTIER) // 14)
print(f'every {step}th frontier point, cheapest first:' if step > 1
      else 'the frontier, cheapest first:')
print(f'{"cost/req":>10s} {"accuracy":>9s} {"escalated":>10s}  configuration')
for p in FRONTIER[::step] + ([FRONTIER[-1]] if step > 1 else []):
    print(f'${p["cost"]:9.6f} {p["accuracy"]:9.3f} {p["escalation"]:10.1%}  '
          f'{p["label"]}')

In [ ]:
def tally(points):
    out = {}
    for p in points:
        out[p['label']] = out.get(p['label'], 0) + 1
    return sorted(out.items(), key=lambda kv: -kv[1])


print('who owns the frontier, by point count:')
for label, count in tally(FRONTIER):
    print(f'  {count:3d} point(s)  {label}')

# Counting points is a tempting summary and a misleading one, because the cheap end
# of the frontier is densely populated and the expensive end is not. Ownership is
# regional, and the region that matters is the one at or above the quality you have
# to ship, which is the strong model's accuracy.
high = [p for p in FRONTIER if p['accuracy'] >= acc_strong]
print(f'\nwho owns the frontier at or above the strong baseline '
      f'({acc_strong:.3f} accuracy), which is the only region where parity lives:')
if not high:
    print('  nobody: no cascade on the frontier matches the strong model')
for label, count in tally(high):
    print(f'  {count:3d} point(s)  {label}')
print()
print('Now compare both lists against the ECE table. Calibration error scores a')
print('signal; a frontier scores a system, and a system has to pay for its signal.')

In [ ]:
def strictly_dominated(points):
    """Points beaten outright. An exact tie is a duplicate, not a dominance: two
    thresholds that escalate the same items are one system listed twice, and
    counting them as dominated inflates the number and names a nonsense rival."""
    return [p for p in points
            if any((q['cost'] <= p['cost'] and q['accuracy'] > p['accuracy'])
                   or (q['cost'] < p['cost'] and q['accuracy'] >= p['accuracy'])
                   for q in points)]


# These are the backward bend, and they are worth naming, because they are exactly
# the thresholds that a monotonic reading of the curve would recommend.
for key, label, _ in CONFIGS:
    bad = strictly_dominated(SWEEPS[key])
    if not bad:
        print(f'{label}: no dominated thresholds in its own sweep')
        continue
    worst = max(bad, key=lambda p: max(q['accuracy'] for q in SWEEPS[key]
                                       if q['cost'] <= p['cost']) - p['accuracy'])
    rival = max((q for q in SWEEPS[key] if q['cost'] <= worst['cost']),
                key=lambda q: q['accuracy'])
    print(f'{label}: {len(bad)} of {len(SWEEPS[key])} thresholds dominated. Worst is '
          f'threshold {worst["threshold"]:.2f}, paying ${worst["cost"]:.6f} for '
          f'accuracy {worst["accuracy"]:.3f} when ${rival["cost"]:.6f} already buys '
          f'{rival["accuracy"]:.3f} at threshold {rival["threshold"]:.2f}.')

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for (key, label, _), marker in zip(CONFIGS, ['o', 'D', 's']):
    data = SWEEPS[key]
    ax.plot([d['cost'] for d in data], [d['accuracy'] for d in data],
            marker=marker, ms=3.2, lw=1.1, alpha=0.85, label=label)

ax.plot([p['cost'] for p in FRONTIER], [p['accuracy'] for p in FRONTIER],
        color='black', lw=2.2, alpha=0.4, zorder=2,
        label=f'Pareto frontier ({len(FRONTIER)} points)')
for base, mk, col in zip(BASELINES, ['v', '^', 'P', '*'],
                         ['tab:green', 'tab:olive', 'tab:brown', 'tab:red']):
    ax.scatter([base['cost']], [base['accuracy']], s=160 if mk == '*' else 105,
               marker=mk, zorder=5, color=col, label=base['label'])
ax.axhline(acc_strong, ls=':', lw=1, color='tab:red')
ax.set_xscale('log')
ax.set_xlabel('cost per request, USD (log scale)')
ax.set_ylabel('accuracy')
ax.set_title('Cost versus accuracy: three routers against four single-model baselines')
ax.legend(loc='lower right', fontsize=7.5)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## The number that justifies the project: savings at quality parity

"Cheaper" is not a result on its own, and neither is "almost as good". The decision
needs one number: holding quality at the strong model's level, what does the cascade
cost? Anything else compares two systems that do different jobs.

Parity is strict here, accuracy greater than or equal to the strong baseline. The
near-parity lines underneath, at one and three accuracy points below, are reported
separately rather than blended in, because the gap between strict parity and a
three-point tolerance is usually where the real decision lives, and quoting only the
looser number is the most common way a cascade gets oversold.

In [ ]:
def cheapest_at(points, floor):
    feasible = [p for p in points if p['accuracy'] >= floor]
    return min(feasible, key=lambda p: p['cost']) if feasible else None


TOLERANCES = [('strict parity', 0.00), ('within 1 point', 0.01),
              ('within 3 points', 0.03)]
print(f'strong baseline: accuracy {acc_strong:.3f} at ${C_STRONG:.6f} per request\n')
print(f'{"router":22s} {"tolerance":16s} {"cost/req":>10s}  {"vs always-strong":>19s}'
      f'  {"accuracy":>8s} {"escal.":>7s}')
for key, label, _ in CONFIGS:
    for name, give in TOLERANCES:
        best = cheapest_at(SWEEPS[key], acc_strong - give)
        if best is None:
            print(f'{label:22s} {name:16s} {"unreachable":>10s}')
            continue
        print(f'{label:22s} {name:16s} ${best["cost"]:9.6f}  '
              f'{verdict(best["cost"]):>19s}  {best["accuracy"]:8.3f} '
              f'{best["escalation"]:7.1%}')
    print()

strict = [cheapest_at(SWEEPS[k], acc_strong) for k, _, _ in CONFIGS]
strict = [p for p in strict if p is not None]
if strict:
    win = min(strict, key=lambda p: p['cost'])
    print(f'Headline: at strict parity the cheapest cascade is {win["label"]}, at '
          f'${win["cost"]:.6f}\nper request, which is {verdict(win["cost"]).strip()} '
          f'than always-strong.')
else:
    print('Headline: no router reaches strict parity. The honest report is that this')
    print('cascade cannot match the strong model at any threshold.')

## Is the confidence signal doing the work?

Two controls, because beating the weak baseline proves nothing: sending a random 40%
of traffic to a better model also improves accuracy.

**The random router** escalates the same share of items, chosen at random. The gap
between it and the real router is what the confidence signal is worth, and it is the
only comparison that holds the escalation budget fixed while varying the signal.

**The oracle router** escalates exactly the items the weak tier got wrong. It is not
implementable, which is the point: it is the ceiling on every possible confidence
signal, and the distance from the real router up to it is the headroom a better signal
could still recover.

In [ ]:
def random_router(rows, key, rate, k, seed=3):
    r = np.random.default_rng(seed)
    escalate = r.random(len(rows)) < rate
    correct = [row['strong_ok'] if e else row[f'{key}_ok']
               for row, e in zip(rows, escalate)]
    return float(np.mean(correct)), cascade_cost(float(np.mean(escalate)), k=k)


def oracle_router(rows, key, k):
    escalate = [not row[f'{key}_ok'] for row in rows]
    correct = [row['strong_ok'] if e else row[f'{key}_ok']
               for row, e in zip(rows, escalate)]
    rate = float(np.mean(escalate))
    return float(np.mean(correct)), cascade_cost(rate, k=k), rate


print(f'{"router":34s} {"escal.":>8s} {"accuracy":>9s} {"cost/req":>10s}   note')
for key, label, k in CONFIGS:
    target = cheapest_at(SWEEPS[key], acc_strong - 0.01) or SWEEPS[key][-1]
    print(f'{label + ", at parity-1pt":34s} {target["escalation"]:8.1%} '
          f'{target["accuracy"]:9.3f} ${target["cost"]:9.6f}')
    r_acc, r_cost = random_router(table, key, target['escalation'], k)
    print(f'{"  random, same escalation rate":34s} {target["escalation"]:8.1%} '
          f'{r_acc:9.3f} ${r_cost:9.6f}   signal worth '
          f'{target["accuracy"] - r_acc:+.3f}')
    o_acc, o_cost, o_rate = oracle_router(table, key, k)
    print(f'{"  oracle (unimplementable)":34s} {o_rate:8.1%} {o_acc:9.3f} '
          f'${o_cost:9.6f}   headroom {o_acc - target["accuracy"]:+.3f}')

## The savings depend on the price ratio, not the prices

Vendor prices change, often sharply, and a cascade justified at one ratio is pointless
at another. Sweeping `c_s / c_w` is how you find out whether the design survives the
next pricing page, and it costs nothing: the accuracy table is already measured, so
only the arithmetic changes.

The breakeven ratio is searched for rather than stated. An earlier draft of this
notebook asserted one from intuition and its own sweep disagreed by a wide margin,
which is the general argument for computing the thing you are about to claim.

In [ ]:
def best_saving_at_ratio(ratio, key, k, give=0.01):
    c_s = C_WEAK * ratio
    rescored = [{**p, 'cost': cascade_cost(p['escalation'], k=k, c_s=c_s)}
                for p in SWEEPS[key]]
    best = cheapest_at(rescored, acc_strong - give)
    return (None, None) if best is None else (best, 1.0 - best['cost'] / c_s)


short = {key: label.replace('self-consistency ', 'sc ').replace('logprob proxy ', 'lp ')
         for key, label, _ in CONFIGS}
print('saving vs always-strong, holding accuracy within 1 point of it')
print(f'{"c_s/c_w":>8s}' + ''.join(f'{short[k]:>14s}' for k, _, _ in CONFIGS))
for ratio in [2, 5, 10, 13, 17, 30, 60, 120]:
    line = f'{ratio:8d}'
    for key, _, k in CONFIGS:
        _, saving = best_saving_at_ratio(ratio, key, k)
        line += f'{"n/a" if saving is None else format(saving, "+.1%"):>14s}'
    print(line)
print()
print('A negative entry means the cascade costs MORE than simply always calling the')
print('strong model, which is the normal situation when the tiers are close in price.')

In [ ]:
print('breakeven: the lowest price ratio at which the cascade is cheaper than')
print('always-strong while holding accuracy within 1 point of it.\n')
grid = np.arange(1.5, 80.01, 0.1)
for key, label, k in CONFIGS:
    hit = next((r for r in grid
                if (best_saving_at_ratio(r, key, k)[1] or -1.0) > 0), None)
    if hit is None:
        print(f'  {label:22s} never breaks even below 80x')
    else:
        print(f'  {label:22s} breaks even at {hit:.1f}x')
print(f'\nToday\'s measured ratio is {C_STRONG / C_WEAK:.1f}x.')
print('Read each breakeven against that number. A configuration whose breakeven sits')
print('just below it is one pricing update away from being a liability, and k is the')
print('lever: every extra confidence sample multiplies the cheap tier and so pushes')
print('that configuration\'s breakeven ratio up.')

## Observations to notice

1. **The weak call is never refunded, and that sets a hard floor.** Cost is
   `k * c_w + e * c_s`, so the cheapest the cascade can ever be is `k * c_w` and the
   most expensive is strictly worse than always-strong. An analysis using
   `(1 - e) * c_w + e * c_s` overstates the saving, and overstates it most where
   escalation is high, which is exactly where the decision is tightest.

2. **Frontier ownership is regional, so "who owns the frontier" is a misleading
   summary of it.** By point count the free signal wins easily, because the cheap end
   of the frontier is where its continuous threshold can place many closely spaced
   points. Restrict to the region at or above the strong baseline's accuracy, which is
   the only region where parity is even possible, and the self-consistency routers own
   it instead. Both lists are printed for exactly this reason: a single count over an
   unevenly populated frontier answers a question nobody asked.

3. **Calibration error ranks signals, not systems.** ECE is worth computing and worth
   nothing as a selection criterion. It cannot see the `k` the signal charges, it
   cannot see where on the cost axis a router operates, and it has no opinion about
   which region of the frontier you need. Rank routers by the frontier, decide with
   the parity table, and use calibration for what it is genuinely diagnostic of,
   which is whether the threshold axis means anything at all.

4. **Majority voting raises accuracy before any routing happens.** Part of what looks
   like a routing win in a `k`-sample cascade is just the vote, which is why the
   `always weak, k samples` markers are on the chart. Measuring a self-consistency
   cascade against the 1-sample weak baseline credits the router with the vote's work,
   and the printed accuracy deltas say exactly how much credit that is.

5. **A sweep is not a frontier: test for dominance.** The dominated-point cell names
   thresholds that cost more and deliver less than a cheaper setting in the same
   sweep. They exist because escalating an item the weak tier got right and the strong
   tier gets wrong loses on both axes at once, so the curve bends backwards at the
   aggressive end. Eyeballing the chart will not catch this; the dominance test will.
   Note which sweep reports them and which do not, and read that against the next
   observation: a signal with only a handful of reachable operating points has too
   few rungs to bend, so the finer your threshold control, the more carefully you
   have to check that you have not tuned your way into a strictly worse system.

6. **Agreement-share confidence is coarse, and coarseness costs you control.** A
   `k`-sample signal has only `k + 1` possible values, so forty-one thresholds collapse
   onto a handful of reachable escalation rates, as the distinct-value lists show. The
   free continuous signal can be tuned to whatever budget you have; the expensive
   discrete one offers a short menu, and if your target escalation rate falls between
   two rungs, no threshold will give it to you.

7. **Compare against the random router, always.** Escalating a random share of traffic
   to a better model improves accuracy for free, so the gap to random is the value of
   the confidence signal and the gap to oracle is the headroom left. A cascade that
   only beats the weak baseline has demonstrated nothing about its router, and both
   numbers are printed beside every configuration for that reason.

8. **The price ratio decides the design, and the breakeven is searched for rather than
   assumed.** Savings collapse as the tiers converge in price, and the breakeven cell
   reports the exact ratio each configuration needs. Note how much higher the
   breakeven runs for configurations that buy more confidence samples: `k` multiplies
   the cheap tier, so it pushes the breakeven up directly. Re-run that cell when
   prices move; the accuracy table does not need re-measuring.

9. **Every accuracy number here is a property of the simulator.** The gap shape, the
   distractor skew and the confidence noise are stated assumptions, and replacing them
   with your own data is the first thing to do. What is *not* simulated is the
   arithmetic: the cost formula, the dominance test, the parity search and the
   breakeven search are exact, and those are the parts that transfer.

## The live version

Everything above is reproducible and free. This section replaces the simulator with
gpt-4o-mini and gpt-4o on the same generated benchmark, and it is the only code here
that spends money. Set `RUN_LIVE = True` in the setup cell to enable it.

Cost scales as `n * k` weak calls plus the escalations. Start at `n = 40`, read the
printed estimate before committing to anything larger, and treat the result as a
smoke test of the pipeline rather than a measurement of either model: forty items
cannot separate two models whose accuracies differ by a few points.

In [ ]:
# SKIP-LOCAL: calls gpt-4o-mini and gpt-4o per item, so it costs real money
def live_answer(item, model, temperature):
    from openai import OpenAI
    client = OpenAI()
    prompt = (f'Compute {item["text"]}. Reply with a single JSON object: '
              '{"answer": <integer>}. No other text.')
    response = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{'role': 'user', 'content': prompt}])
    try:
        return int(json.loads(response.choices[0].message.content)['answer'])
    except (json.JSONDecodeError, KeyError, TypeError, ValueError):
        return None


def live_table(items, k=3):
    rows = []
    for item in items:
        samples = [s for s in (live_answer(item, 'gpt-4o-mini', 1.0)
                               for _ in range(k)) if s is not None]
        if not samples:
            continue   # every sample came back unparseable; nothing to route on
        values, counts = np.unique(samples, return_counts=True)
        best = int(np.argmax(counts))
        strong = live_answer(item, 'gpt-4o', 0.0)
        rows.append({'truth': item['answer'], 'difficulty': item['difficulty'],
                     f'sc{k}_answer': int(values[best]),
                     f'sc{k}_conf': counts[best] / k,
                     f'sc{k}_ok': int(values[best]) == item['answer'],
                     'strong_ok': strong == item['answer']})
    return rows

In [ ]:
# SKIP-LOCAL: spends money; gated on RUN_LIVE so opening the notebook is free
if RUN_LIVE:
    subset, k_live = ITEMS[:40], 3
    print(f'about to spend roughly '
          f'${len(subset) * (k_live * C_WEAK + C_STRONG):.4f} on {len(subset)} items')
    live_rows = live_table(subset, k=k_live)
    live_strong = float(np.mean([r['strong_ok'] for r in live_rows]))
    live_weak = float(np.mean([r[f'sc{k_live}_ok'] for r in live_rows]))
    live_sweep = sweep(live_rows, f'sc{k_live}', k_live, 'live')
    print(f'\nlive rows: {len(live_rows)} of {len(subset)}')
    print(f'weak (majority of {k_live}) accuracy {live_weak:.3f}')
    print(f'strong accuracy                {live_strong:.3f}')
    # The simulator's conclusions only transfer if the real gap has the same sign and
    # rough size. If the real tiers tie, no router can save you anything at all.
    print(f'measured gap {live_strong - live_weak:+.3f}, against a simulated gap of '
          f'{acc_strong - acc[f"sc{k_live}"]:+.3f}')
    best = cheapest_at(live_sweep, live_strong - 0.01)
    if best is None:
        print('no threshold reaches parity-1pt on this subset')
    else:
        print(f'parity-1pt at ${best["cost"]:.6f} per request, '
              f'{verdict(best["cost"])}, escalating {best["escalation"]:.0%}')
    for p in pareto(live_sweep):
        print(f'  frontier: ${p["cost"]:.6f}  acc {p["accuracy"]:.3f}  '
              f'escalating {p["escalation"]:.0%}')
else:
    print('RUN_LIVE is False, so no API calls were made.')

## What to try next

- Replace the generated benchmark with your own traffic, keeping the verifier. The
  difficulty distribution of real requests is the single biggest input to a cascade's
  economics, and it is never uniform the way this one is.
- Add a third tier. The formula generalises to `k*c_1 + e_1*c_2 + e_1*e_2*c_3`, and
  the compounding escalation term is where three-tier designs quietly stop paying. Run
  the dominance test on the result; more tiers produce more dominated configurations,
  not fewer.
- Route on a cheap learned signal instead: train a small classifier on request
  features to predict whether the weak tier will be right, and put its curve on these
  same axes. It costs one forward pass, so it lands beside the free signal on the cost
  axis while potentially carrying far more information than either signal here.
- Add latency to the objective. An escalation is two sequential calls, so p95 latency
  degrades precisely on the hard requests, and a cost win can be a user experience
  loss that the cost-accuracy plane cannot show you.
- Measure the cost of being wrong. Parity assumes an error costs the same whoever made
  it; if a wrong answer costs a support ticket, fold that into the objective and watch
  the optimal threshold move.
- Re-derive the frontier with per-item costs rather than a fixed token count. Long
  requests are exactly the ones that escalate, so the average escalated item costs more
  than the average item and a fixed `c_s` understates the bill.
- Sweep the *accuracy gap* as well as the price ratio. If the tiers are close, no
  router helps; the gap and the ratio together decide whether a cascade is a project
  or a distraction, and each is one parameter in the code above.